# 너비 우선 탐색 (Breadth-First Search)

`-` 시작 노드에서 가까운 노드부터 차례대로 탐색하는 알고리즘

## 보물섬

- 문제 출처: [정올 1462번](https://jungol.co.kr/problem/1462)

`-` 각 육지 칸마다 BFS를 수행해서 가장 먼 육지 칸까지의 이동 시간을 계산하자

`-` 이들 중 최댓값이 문제의 정답이 된다

`-` 전체 알고리즘의 시간 복잡도는 모든 칸이 육지인 최악의 경우 $O\left(R^2C^2\right)$이다

In [6]:
from collections import deque
from itertools import product


def find_land_positions(graph, n_rows, n_cols):
    return [(r, c) for r, c in product(range(n_rows), range(n_cols)) if graph[r][c] == LAND]


def bfs(graph, r_start, c_start, n_rows, n_cols):
    pos_start = r_start, c_start
    queue = deque([pos_start])
    visited = {pos_start: 0}
    drc = [(0, -1), (0, 1), (-1, 0), (1, 0)]
    while queue:
        r, c = queue.popleft()
        for dr, dc in drc:
            nr, nc = r + dr, c + dc
            is_in_range = 0 <= nr < n_rows and 0 <= nc < n_cols
            if not is_in_range or graph[nr][nc] != LAND:
                continue
            if (nr, nc) in visited:
                continue
            visited[nr, nc] = visited[r, c] + 1
            queue.append((nr, nc))
    return visited


def solution():
    global LAND
    R, C = map(int, input().split())
    graph = [input() for _ in range(R)]
    LAND = "L"
    land_positions = find_land_positions(graph, R, C)
    answer = 0
    for r, c in land_positions:
        pos2time = bfs(graph, r, c, R, C)
        time = max(pos2time.values())
        answer = max(time, answer)
    print(answer)


solution()

# input
# 2 2
# LL
# LL

 2 2
 LL
 LL


2


## 치즈

- 문제 출처: [정올 1840번](https://jungol.co.kr/problem/1840)

`-` 0-1 BFS로 풀 수 있다고 한다

`-` 치즈가 외부 공기와 접촉하면 한 시간 후에 녹아 없어진다

`-` 공기로 이동할 땐 가중치를 $0$으로 두고 치즈로 이동할 땐 가중치를 $1$로 두자

`-` 그럼 공기로만 이루어진 컴포넌트를 $0$ 시간에 구성한 뒤 치즈와의 접촉 여부를 판단할 수 있게 된다

`-` 시작 지점은 항상 외부 공기임이 보증된 $(0,0)$이며 가중치가 $0$과 $1$뿐이니 0-1 BFS를 사용하여 $O(V+E)$에 해결할 수 있다

`-` 거리 배열의 최댓값이 치즈가 모두 녹아 없어지는 데 걸리는 시간이며 치즈가 있으면서 거리가 최댓값인 칸의 개수가 녹기 직전 칸의 개수이다

In [7]:
from collections import deque


def bfs(graph, source):
    n_rows, n_cols = len(graph), len(graph[0])
    queue = deque([source])
    visited = [[-1] * n_cols for _ in range(n_rows)]
    visited[source[0]][source[1]] = 0
    drc = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    while queue:
        r, c = queue.popleft()
        for dr, dc in drc:
            nr, nc = r + dr, c + dc
            is_in_range = 0 <= nr < n_rows and 0 <= nc < n_cols
            if not is_in_range or visited[nr][nc] >= 0:
                continue
            is_cheese = graph[nr][nc] == 1
            if is_cheese:
                queue.append((nr, nc))
                visited[nr][nc] = visited[r][c] + 1
            else:
                queue.appendleft((nr, nc))
                visited[nr][nc] = visited[r][c]
    return visited


def solution():
    R, C = map(int, input().split())
    graph = [list(map(int, input().split())) for _ in range(R)]
    source = 0, 0
    visited = bfs(graph, source)
    end_time = max(map(max, visited))
    count = sum(1 for r in range(R) for c in range(C) if visited[r][c] == end_time and graph[r][c] == 1)
    print(end_time)
    print(count)


solution()

# input
# 3 3
# 0 0 0
# 0 1 0
# 0 0 0

 3 3
 0 0 0
 0 1 0
 0 0 0


1
1


`-` 근데 꼭 가중치가 $0$과 $1$일 필요는 없고 $0$과 $c$여도 괜찮다

`-` 하지만 $0$이 아닌 $a,b$라면 가중치가 $a$인 간선을 우선 사용하는 게 항상 최단 거리를 보장하는 게 아니게 된다

## 치즈

- 문제 출처: [정올 1870번](https://jungol.co.kr/problem/1870)

`-` 이전 [치즈](https://jungol.co.kr/problem/1840) 문제와의 차이점은 치즈가 있는 칸의 $2$변 이상이 외부 공기와 접촉해야 치즈가 녹는다는 것이다 (이전 문제에선 $1$변)

`-` 0-1 BFS를 수행하며 치즈로 이동할 때 카운팅을 하자. 만약 카운팅이 $2$가 됐다면 큐에 넣고 방문 표시를 하면 된다

In [8]:
from collections import deque


def bfs(graph, source):
    n_rows, n_cols = len(graph), len(graph[0])
    queue = deque([source])
    visited = [[-1] * n_cols for _ in range(n_rows)]
    visited[source[0]][source[1]] = 0
    counts = [[0] * n_cols for _ in range(n_rows)]
    drc = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    while queue:
        r, c = queue.popleft()
        for dr, dc in drc:
            nr, nc = r + dr, c + dc
            is_in_range = 0 <= nr < n_rows and 0 <= nc < n_cols
            if not is_in_range or visited[nr][nc] >= 0:
                continue
            is_cheese = graph[nr][nc] == 1
            if is_cheese:
                counts[nr][nc] += 1
                if counts[nr][nc] < 2:
                    continue
                queue.append((nr, nc))
                visited[nr][nc] = visited[r][c] + 1
            else:
                queue.appendleft((nr, nc))
                visited[nr][nc] = visited[r][c]
    return visited


def solution():
    R, C = map(int, input().split())
    graph = [list(map(int, input().split())) for _ in range(R)]
    source = 0, 0
    visited = bfs(graph, source)
    end_time = max(map(max, visited))
    print(end_time)


solution()

# input
# 3 3
# 0 0 0
# 0 1 0
# 0 0 0

 3 3
 0 0 0
 0 1 0
 0 0 0


1


## 화염에서탈출

- 문제 출처: [정올 1082번](https://jungol.co.kr/problem/1082)

`-` 불이든 재우든 `S`, `*`, `X`인 칸에는 갈 필요가 없다

`-` 불이 먼저 이동한 뒤 재우가 이동한다. 따라서 불의 좌표와 재우가 처음 서 있는 위치를 큐에 넣자. 불이 큐에서 먼저 나오면서 그래프를 갱신하게 된다

`-` 큐에서 나온 원소가 재운인 경우 거리 배열을 갱신하고 다음에 이동할 칸을 `S`로 갱신하자

`-` 불인 경우 다음에 이동할 칸만 `*`로 갱신하자

`-` 같은 좌표를 $2$번 이상 큐에 넣지 않으므로 전체 알고리즘의 시간 복잡도는 $O(RC)$이다

`-` 만약 재우가 움직일 때마다 불이 옮기는 것을 반영하고자 이중 for문을 돌며 인접한 $4$칸 중 최소 한 곳에 불이 있는지 확인하는 건 최악의 경우 $O\left(R^2 C^2\right)$의 시간 복잡도를 가진다

In [2]:
from collections import deque


def find_positions(graph, mark):
    n_rows, n_cols = len(graph), len(graph[0])
    return [(r, c) for r in range(n_rows) for c in range(n_cols) if graph[r][c] == mark]


def bfs(graph, source, sink, flames):
    n_rows, n_cols = len(graph), len(graph[0])
    queue = deque(flames + [source])
    distances = [[-1] * n_cols for _ in range(n_rows)]
    distances[source[0]][source[1]] = 0
    drc = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    while queue:
        r, c = queue.popleft()
        for dr, dc in drc:
            nr, nc = r + dr, c + dc
            is_in_range = 0 <= nr < n_rows and 0 <= nc < n_cols
            if not is_in_range:
                continue
            if graph[nr][nc] != "D" and graph[nr][nc] != ".":
                continue
            is_man = graph[r][c] == "S"
            if is_man:
                queue.append((nr, nc))
                graph[nr][nc] = "S"
                distances[nr][nc] = distances[r][c] + 1
            elif graph[nr][nc] == ".":
                queue.append((nr, nc))
                graph[nr][nc] = "*"
    if distances[sink[0]][sink[1]] == -1:
        return "impossible"
    return distances[sink[0]][sink[1]]


def solution():
    R, C = map(int, input().split())
    graph = [list(input().rstrip()) for _ in range(R)]
    source = find_positions(graph, "S")[0]
    sink = find_positions(graph, "D")[0]
    flames = find_positions(graph, "*")
    answer = bfs(graph, source, sink, flames)
    print(answer)


solution()

# input
# 2 2
# DS
# ..

 2 2
 DS
 ..


1


## 가로등

- 문제 출처: [정올 8017번](https://jungol.co.kr/problem/8017)

`-` 멀티 소스 BFS를 수행해서 밝은 순서대로 $K$개의 위치를 방문하자

`-` 전체 알고리즘의 시간 복잡도는 $O(N+K)$이다

In [1]:
from collections import deque


def bfs(sources, limit, k):
    n = len(sources)
    if n >= k:
        return [0] * k
    k -= n
    queue = deque(sources)
    visited = {}
    for s in sources:
        visited[s] = 0
    values = [0] * n
    while queue and k > 0:
        u = queue.popleft()
        for dx in [1, -1]:
            v = u + dx
            if not (0 <= v <= limit):
                continue
            if v in visited or k <= 0:
                continue
            queue.append(v)
            visited[v] = visited[u] + 1
            values.append(visited[v])
            k -= 1
    return values


def solution():
    L, N, K = map(int, input().split())
    sources = list(map(int, input().split()))
    answer = bfs(sources, L, K)
    print("\n".join(map(str, answer)))


solution()

# input
# 5 1 2
# 3

 5 1 2
 3


0
1


## 경로 찾기

- 문제 출처: [정올 2261번](https://jungol.co.kr/problem/2261)

`-` 이진수를 그래프 상 노드라 생각하면 이진수 하나당 가능한 간선의 최대 개수는 $K$이다 (비트 하나만 바꿀 수 있는데 비트 수가 총 $K$개이기 때문이다) 

`-` 노드의 개수는 $N$을 넘지 못하므로 그래프 상 간선의 개수는 $O(NK)$이다. 그런데 현재 노드에서 가능한 다음 노드는 $O\left(K^2\right)$에 얻을 수 있으므로 BFS의 시간 복잡도는 $O\left(NK^2\right)$이다

In [4]:
from collections import defaultdict, deque


def bfs(bin2id, id2bin, source, sink):
    queue = deque([source])
    visited = {source: 0}
    predecessors = defaultdict(lambda: None)
    while queue:
        u = queue.popleft()
        u_bin = id2bin[u]
        next_nodes = find_next_nodes(u_bin)
        for v_bin in next_nodes:
            if v_bin not in bin2id or bin2id[v_bin] in visited:
                continue
            v = bin2id[v_bin]
            queue.append(v)
            visited[v] = visited[u] + 1
            predecessors[v] = u
            if v == sink:
                return predecessors
    return -1


def find_next_nodes(binary):
    nodes = []
    for i in range(len(binary)):
        node = list(binary)
        node[i] = 1 - node[i]
        nodes.append(tuple(node))
    return nodes


def track(predecessors, source, sink):
    route = []
    node = sink
    while predecessors[node] is not None:
        route.append(node)
        node = predecessors[node]
    route.append(source)
    route = route[::-1]
    return route


def solution():
    N, K = map(int, input().split())
    bin2id = {}
    id2bin = {}
    for i in range(1, N + 1):
        binary = tuple(map(int, list(input().rstrip())))
        bin2id[binary] = i
        id2bin[i] = binary
    A, B = map(int, input().split())
    predecessors = bfs(bin2id, id2bin, A, B)
    if predecessors == -1:
        print(-1)
        return
    route = track(predecessors, A, B)
    print(*route)


solution()

# input
# 5 3
# 000
# 111
# 010
# 110
# 001
# 1 2

 5 3
 000
 111
 010
 110
 001
 1 2


1 3 4 2


## 과학 수행평가

- 문제 출처: [정올 8986번](https://jungol.co.kr/problem/8986)

`-` 핵심은 최소 $1$개의 도체 경로는 남겨야 한다는 것이다. 최단 거리가 $x$일 때 $x+1$개의 도체를 제외하면 나머지 도체는 필요 없다 (간선이 $x$개이므로 노드는 $x+1$개)

`-` 따라서 전체 도체 개수에서 $x+1$을 차감한 것이 최대 점수이다. $x$는 멀티 소스 BFS로 계산할 수 있으며 소스는 $1$행이 아니라 $1$열이고 $c$열이 싱크임에 유의하자

In [1]:
from collections import deque


def bfs(graph, sources, sinks):
    n_rows, n_cols = len(graph), len(graph[0])
    queue = deque(sources)
    distances = [[None] * n_cols for _ in range(n_rows)]
    for r, c in sources:
        distances[r][c] = 0
    drc = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    while queue:
        r, c = queue.popleft()
        for dr, dc in drc:
            nr, nc = r + dr, c + dc
            is_in_range = 0 <= nr < n_rows and 0 <= nc < n_cols
            if not is_in_range:
                continue
            if graph[nr][nc] == "#" or distances[nr][nc] is not None:
                continue
            queue.append((nr, nc))
            distances[nr][nc] = distances[r][c] + 1
            if (nr, nc) in sinks:
                return distances[nr][nc]


def solution():
    R, C = map(int, input().split())
    graph = [list(input().rstrip()) for _ in range(R)]
    sources = [(r, 0) for r in range(R) if graph[r][0] == "."]
    sinks = set([(r, C - 1) for r in range(R) if graph[r][C - 1] == "."])
    total = sum(1 for row in graph for s in row if s == ".")
    min_distance = bfs(graph, sources, sinks)
    answer = total - min_distance - 1
    print(answer)


solution()

# input
# 1 3
# ...

 1 3
 ...


0


## 미로만들기

- 문제 출처: [정올 1696번](https://jungol.co.kr/problem/1696)

`-` 여태까지 바꾼 방의 개수를 변수로서 기록하여 이미 방문한 좌표더라도 여태까지 바꾼 방의 개수를 더 적게 할 수 있다면 방문할 것이다

`-` 전체 알고리즘의 시간 복잡도는 최악의 경우 $O\left(n^4\right)$이다

In [1]:
from collections import deque


def bfs(graph, source):
    n_rows, n_cols = len(graph), len(graph[0])
    queue = deque([(*source, 0)])
    visited = [[n_rows * n_cols] * n_cols for _ in range(n_rows)]
    visited[source[0]][source[1]] = 0
    drc = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    while queue:
        r, c, x = queue.popleft()
        for dr, dc in drc:
            nr, nc = r + dr, c + dc
            is_within_grid = 0 <= nr < n_rows and 0 <= nc < n_cols
            if not is_within_grid:
                continue
            nx = x + (graph[nr][nc] == "0")
            if visited[nr][nc] <= nx:
                continue
            queue.append((nr, nc, nx))
            visited[nr][nc] = nx
    return visited


def solution():
    n = int(input())
    graph = [input().rstrip() for _ in range(n)]
    source, sink = (0, 0), (n - 1, n - 1)
    visited = bfs(graph, source)
    answer = visited[sink[0]][sink[1]]
    print(answer)


solution()

# 2
# 01
# 10

 2
 01
 10


1


## 물통

- 문제 출처: [정올 3078번](https://jungol.co.kr/problem/3078)

`-` 두 물통에 담긴 물의 양을 상태로 하여 BFS를 수행하자. 상태 공간이 희소하여 생각보다 빠른 시간에 동작한다. 상태 공간을 배열로 표현하면 $O(ab)$라 시간 초과이므로 딕셔너리를 사용해야 한다

`-` 상태 공간이 얼마나 희소한지 몰라 찜찜하긴 하지만 맞혔다

In [8]:
from collections import deque


def bfs(a, b, c, d):
    queue = deque([(0, 0)])
    distances = {(0, 0): 0}
    while queue:
        x, y = queue.popleft()
        d_xy = distances[x, y]
        if x == c and y == d:
            return d_xy
        if (a, y) not in distances:
            queue.append((a, y))
            distances[a, y] = d_xy + 1
        if (x, b) not in distances:
            queue.append((x, b))
            distances[x, b] = d_xy + 1
        if (0, y) not in distances:
            queue.append((0, y))
            distances[0, y] = d_xy + 1
        if (x, 0) not in distances:
            queue.append((x, 0))
            distances[x, 0] = d_xy + 1
        d1 = min(x, b - y)
        if (x - d1,  y + d1) not in distances:
            queue.append((x - d1,  y + d1))
            distances[x - d1, y + d1] = d_xy + 1
        d2 = min(y, a - x)
        if (x + d2,  y - d2) not in distances:
            queue.append((x + d2,  y - d2))
            distances[x + d2,  y - d2] = d_xy + 1
    return -1


def solution():
    a, b, c, d = map(int, input().split())
    answer = bfs(a, b, c, d)
    print(answer)


solution()

# input
# 3 7 3 2

 2 5 0 1


5


`-` 역시 이따구로 푸는 문제가 아니었다!

`-` 각 작업을 잘 보면 매 작업 후 가능한 상태는 $4$가지 뿐이다 (A가 가득 참, B가 가득 참, A가 빔, B가 빔). 따라서 상태 공간의 크기는 $O(a + b)$가 된다

## 경비행기

- 문제 출처: [정올 1603번](https://jungol.co.kr/problem/1603)

`-` 결정 문제로 바꿔서 해결하자. 출발지와 목적지 사이의 거리는 대략 $14142$이므로 연료통의 최대 용량은 $1415$이다. 연료통 용량에 따른 최소 중간급유 횟수는 단조성을 가지므로 이분 탐색을 이용할 수 있다. 연료통 용량에 따라 BFS를 수행하면 되며 BFS 호출 횟수는 $O(\log 1415)$이다

`-` 노드가 $O(n)$개, 간선이 $O\left(n^2\right)$개이고 각 위치별로 가능한 급유 횟수가 $k$이므로 BFS의 시간 복잡도는 최악의 경우 $O\left(n^2 k\right)$이고 이는 시간 초과이다. 대신 b번 사항을 고려해 그리디하게 생각하자

`-` 연료통 용량을 $L$이라 할 때 출발지에서 거리가 $10L$ 이하인 지점으로 $1$번에 이동하자. 이들은 $0$번의 중간급유로 이동할 수 있으며 이 외의 지점은 불가능하다. 그리고 또 이미 방문한 지점을 더 적은 중간급유 횟수로 이동하는 것도 불가능하다 (삼각 부등식에 의거). 그럼 각 지점별로 중간급유 횟수가 고정이니 BFS는 최악의 경우에도 $O\left(n^2\right)$에 동작한다

`-` 따라서 전체 알고리즘의 시간 복잡도는 최악의 경우에도 $O\left(n^2 \log 1415\right)$이다

In [2]:
from collections import deque


def binary_search(points, source, sink, k):
    low, high = 1, 1415
    while low <= high:
        mid = (low + high) // 2
        count = bfs(points, mid, source, sink)
        if count <= k:
            high = mid - 1
        else:
            low = mid + 1
    return low


def bfs(points, capacity, source, sink):
    cs = 100 * capacity**2
    queue = deque([source])
    visited = {source: -1}
    while queue:
        x, y = queue.popleft()
        for nx, ny in points:
            if (nx, ny) in visited:
                continue
            if (nx - x)**2 + (ny - y)**2 > cs:
                continue
            visited[nx, ny] = visited[x, y] + 1
            if (nx, ny) == sink:
                return visited[nx, ny]
            queue.append((nx, ny))
    return INF


def solution():
    global INF
    INF = float("inf")
    source, sink = (0, 0), (10000, 10000)
    n, k = map(int, input().split())
    points = [list(map(int, input().split())) for _ in range(n)]
    points.append(sink)
    answer = binary_search(points, source, sink, k)
    print(answer)


solution()

# input
# 1 1
# 5000 5000

 1 1
 5000 5000


708
